# Assignment 3: Build Non-Linear Models Part 1

Juan Maldonado Franco  
DDS-8555 Predictive Analysis  
Mohamed Nabeel

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd()
for parent in [ROOT, *ROOT.parents]:
    if (parent / "DDS-8555 - Predictive Analysis").exists():
        COURSE = parent / "DDS-8555 - Predictive Analysis"
        break
else:
    COURSE = ROOT.parents[1]
DATA = COURSE / "data"
KAGGLE = DATA / "kaggle"
SUBMISSIONS = DATA / "submissions"
RANDOM_STATE = 42
pd.set_option("display.max_columns", 80)

## Conceptual Question 1

For any fixed number of predictors k, best subset selection must have training RSS less than or equal to forward or backward stepwise selection because it searches all k-predictor models.  Test RSS cannot be known from the search method alone because the best training model may overfit.  The forward stepwise nesting statement is true, backward stepwise nesting is true when read in the backward path direction, and the cross-method subset claims are false.  Best subset models are also not necessarily nested across k because the best k-variable model does not have to be contained inside the best k + 1-variable model (James et al., 2023).

## Applied Question 8: Simulated Selection

The simulated exercise compares forward selection, backward selection, and lasso on polynomial features.  The data-generating process includes powers of X, so a good selection method should recover the important polynomial terms more often than noise terms.

In [2]:
from itertools import combinations
import statsmodels.api as sm
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline

rng = np.random.default_rng(RANDOM_STATE)
n = 100
X_raw = rng.normal(size=n)
eps = rng.normal(size=n)
Y = 1 + 2 * X_raw - 1.5 * X_raw**2 + .75 * X_raw**3 + eps
poly = pd.DataFrame({f"X^{i}": X_raw**i for i in range(1, 11)})

def fit_aic(cols):
    Xc = sm.add_constant(poly[list(cols)])
    return sm.OLS(Y, Xc).fit()

def forward_select():
    remaining = list(poly.columns)
    chosen = []
    best_aic = np.inf
    while remaining:
        scores = []
        for col in remaining:
            fit = fit_aic(chosen + [col])
            scores.append((fit.aic, col, fit))
        aic, col, fit = min(scores, key=lambda x: x[0])
        if aic < best_aic:
            chosen.append(col); remaining.remove(col); best_aic = aic; best_fit = fit
        else:
            break
    return chosen, best_fit

forward_cols, forward_fit = forward_select()
lasso = Pipeline([("scale", StandardScaler()), ("model", LassoCV(cv=10, random_state=RANDOM_STATE, max_iter=20000))])
lasso.fit(poly, Y)
coef = pd.Series(lasso.named_steps["model"].coef_, index=poly.columns)
print("Forward selected:", forward_cols)
display(forward_fit.params)
display(coef[coef.abs() > 1e-3].rename("lasso_coefficient"))

Forward selected: ['X^1', 'X^2', 'X^3']


const    1.116665
X^1      1.923477
X^2     -1.694628
X^3      0.844008
dtype: float64

X^1    1.463389
X^2   -1.210939
X^3    1.363408
X^4   -0.109297
Name: lasso_coefficient, dtype: float64

## Kaggle Regularization and PCR

The competition portion uses one regularized model and one principal-components regression model.  Regularization controls coefficient instability, while PCR reduces the predictor space before regression (Jolliffe, 2002; Tibshirani, 1996).

In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.linear_model import ElasticNetCV, Ridge
from sklearn.metrics import mean_squared_log_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

abalone = pd.read_csv(KAGGLE / "playground-series-s4e4" / "train.csv")
X = abalone.drop(columns=["Rings"])
y = abalone["Rings"]
cat = ["Sex"]; num = [c for c in X.columns if c not in cat + ["id"]]
pre = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), cat), ("num", StandardScaler(), num)])
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=.2, random_state=RANDOM_STATE)
models = {
    "Elastic net": Pipeline([("pre", pre), ("model", ElasticNetCV(l1_ratio=[.1,.5,.9], cv=5, random_state=RANDOM_STATE, max_iter=20000))]),
    "PCR ridge": Pipeline([("pre", pre), ("pca", PCA(n_components=.95)), ("model", Ridge(alpha=10))]),
}
rows = []
baseline_pred = np.repeat(y_train.mean(), len(y_valid))
rows.append({"model": "Mean baseline", "validation_rmsle": np.sqrt(mean_squared_log_error(y_valid, baseline_pred))})
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = np.maximum(model.predict(X_valid), 1)
    rows.append({"model": name, "validation_rmsle": np.sqrt(mean_squared_log_error(y_valid, pred))})
validation = pd.DataFrame(rows).sort_values("validation_rmsle")
display(validation)

elastic = models["Elastic net"]
feature_names = elastic.named_steps["pre"].get_feature_names_out()
elastic_coefs = pd.DataFrame({
    "feature": feature_names,
    "coefficient": elastic.named_steps["model"].coef_,
})
elastic_coefs["abs_coefficient"] = elastic_coefs["coefficient"].abs()
display(elastic_coefs.sort_values("abs_coefficient", ascending=False).head(12))
display(pd.DataFrame({
    "model_detail": ["elastic_net_alpha", "elastic_net_l1_ratio", "pcr_components_retained", "pcr_explained_variance"],
    "value": [
        elastic.named_steps["model"].alpha_,
        elastic.named_steps["model"].l1_ratio_,
        models["PCR ridge"].named_steps["pca"].n_components_,
        models["PCR ridge"].named_steps["pca"].explained_variance_ratio_.sum(),
    ],
}))

,model,validation_rmsle
1,Elastic net,0.165582
2,PCR ridge,0.186751
0,Mean baseline,0.290949


,feature,coefficient,abs_coefficient
7,num__Whole weight.1,-3.120836,3.120836
9,num__Shell weight,2.781301,2.781301
6,num__Whole weight,1.457643,1.457643
5,num__Height,0.755684,0.755684
1,cat__Sex_I,-0.683558,0.683558
8,num__Whole weight.2,-0.595796,0.595796
4,num__Diameter,0.571336,0.571336
0,cat__Sex_F,0.016277,0.016277
3,num__Length,-0.001220,0.001220
2,cat__Sex_M,0.000000,0.000000


,model_detail,value
0,elastic_net_alpha,0.002452
1,elastic_net_l1_ratio,0.900000
2,pcr_components_retained,4.000000
3,pcr_explained_variance,0.966408


In [4]:
import re

status_path = SUBMISSIONS / "kaggle_submission_status_playground-series-s4e4.txt"
for encoding in ("utf-16", "utf-8"):
    try:
        text = status_path.read_text(encoding=encoding)
        break
    except UnicodeError:
        continue

records = []
for line in text.splitlines():
    parts = re.split(r"\s{2,}", line.strip())
    if len(parts) >= 7 and parts[0].isdigit() and "A3_" in parts[1]:
        records.append({
            "ref": parts[0],
            "fileName": parts[1],
            "date": parts[2],
            "description": parts[3],
            "status": parts[4],
            "publicScore": parts[5],
            "privateScore": parts[6],
        })

display(pd.DataFrame(records))
display(pd.DataFrame({
    "submission_file": ['A3_elastic_net_playground_series_s4e4.csv', 'A3_pcr_ridge_playground_series_s4e4.csv'],
    "exists_locally": [(SUBMISSIONS / file).exists() for file in ['A3_elastic_net_playground_series_s4e4.csv', 'A3_pcr_ridge_playground_series_s4e4.csv']],
}))
display(pd.DataFrame({
    "evidence": ["Public GitHub repository", "Notebook path in repository"],
    "value": [
        "https://github.com/maldo81/dds-8555-predictive-analysis",
        "Week 3/Assignment 3/MaldonadoJDDS8555-3.ipynb",
    ],
}))

,ref,fileName,date,description,status,publicScore,privateScore
0,52966794,A3_pcr_ridge_playground_series_s4e4.csv,2026-05-23 21:27:17.190000,DDS-8555 A3 principal components ridge,SubmissionStatus.COMPLETE,0.18220,0.18228
1,52966791,A3_elastic_net_playground_series_s4e4.csv,2026-05-23 21:27:14.550000,DDS-8555 A3 elastic net regularization,SubmissionStatus.COMPLETE,0.16444,0.16373


,submission_file,exists_locally
0,A3_elastic_net_playground_series_s4e4.csv,True
1,A3_pcr_ridge_playground_series_s4e4.csv,True


,evidence,value
0,Public GitHub repository,https://github.com/maldo81/dds-8555-predictive...
1,Notebook path in repository,Week 3/Assignment 3/MaldonadoJDDS8555-3.ipynb


## Interpretation

The regularized model is the better fit for this stage of the assignment because it keeps the original predictors visible while shrinking unstable coefficients.  The elastic net coefficient table shows which standardized physical measurements retain signal after shrinkage, while the selected alpha and l1 ratio document the regularization strength chosen by cross-validation (Tibshirani, 1996).  PCR is useful as a dimensionality-reduction comparison, but its components are less directly interpretable because the predictors are rotated into principal components (Jolliffe, 2002).  The Kaggle evidence shows both submissions completed, and the public score favored the elastic net model over the PCR ridge model.  The Abalone context matters because rings are a biological age proxy, so the model should be interpreted as a measurement-based approximation rather than a causal aging model (Nash et al., 1994).

## References

Hastie, T., Tibshirani, R., & Friedman, J. (2009). *The elements of statistical learning: Data mining, inference, and prediction* (2nd ed.).  Springer. https://doi.org/10.1007/978-0-387-84858-7

Jolliffe, I.  T. (2002). *Principal component analysis* (2nd ed.).  Springer. https://doi.org/10.1007/b98835

Nash, W.  J., Sellers, T.  L., Talbot, S.  R., Cawthorn, A.  J., & Ford, W.  B. (1994). *The population biology of abalone (Haliotis species) in Tasmania.  I.  Blacklip abalone (H. rubra) from the north coast and islands of Bass Strait* (Technical Report No.  48).  Sea Fisheries Division.

Tibshirani, R. (1996).  Regression shrinkage and selection via the lasso. *Journal of the Royal Statistical Society: Series B, 58*(1), 267-288. https://doi.org/10.1111/j.2517-6161.1996.tb02080.x